# Tutorial: `context_v3` initialization from raw data

This notebook presents a short, tutorial-style workflow focused only on the initialization step.

It shows how to:
- initialize a `WorkflowContext` from a raw-data directory using `context_v3`,
- inspect detected runs and configurations,
- inspect the dedicated reference stores filled during initialization,
- review warnings and basic sanity checks.

The idea is to keep the notebook focused on a single stage: **load and inspect**.


## 1. Environment setup

This first cell moves the notebook to the repository root and adds `src/` to the `PYTHONPATH`.
This is useful whether the notebook is launched from `notebooks/` or from the repository root.


In [ ]:
from pathlib import Path
import os
import sys

%matplotlib inline

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

os.chdir(ROOT)
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"Repository root: {ROOT}")
print(f"Python path includes: {SRC}")


## 2. Define the working paths

Here we define:
- the raw-data directory,
- the output directory used for converted files,
- the instrument name passed to the raw-data converter.


In [ ]:
from scarlet.workflow.context_v3 import initialize_workflow_context_from_raw_directory

RAW_DIR = ROOT / "data" / "SANSLLB" / "raw"
OUTPUT_DIR = ROOT / "data" / "SANSLLB" / "out_context_v3_init"
INSTRUMENT_NAME = "sansllb"

RAW_DIR, OUTPUT_DIR


## 3. Initialize the `WorkflowContext`

This step scans the raw files, converts them to the SCARLET raw format,
detects runs, builds instrumental configurations, and stores everything in a `WorkflowContext` object.


In [ ]:
w = initialize_workflow_context_from_raw_directory(
    RAW_DIR,
    instrument_name=INSTRUMENT_NAME,
    output_dir=OUTPUT_DIR,
    overwrite=True,
)

print(f"runs: {len(w.runs)}")
print(f"configurations: {len(w.configurations)}")
print(f"issues: {len(w.issues)}")
print(f"artifacts: {len(w.artifacts)}")
print(f"output_dir: {w.output_dir}")


## 4. Inspect the detected runs

The cell below gives a compact view of the run registry created during initialization.
This is the first place to verify that `config_id`, `entity`, `mode`, and `sample_name` look correct.


In [ ]:
run_rows = [
    {
        "config_id": key.config_id,
        "entity": key.entity,
        "mode": key.mode,
        "sample_name": key.sample_name,
        "path": str(path),
    }
    for key, path in sorted(
        w.runs.items(),
        key=lambda item: (
            item[0].config_id,
            item[0].entity,
            item[0].mode,
            item[0].sample_name or "",
        ),
    )
]

run_rows[:20]


## 5. Inspect the detected configurations

This summary helps verify that raw files have been grouped into sensible instrumental configurations.


In [ ]:
config_rows = []
for config_id, configuration in sorted(w.configurations.items()):
    config_rows.append(
        {
            "config_id": config_id,
            "wavelength": getattr(configuration, "wavelength", None),
            "sample_detector_distance": getattr(configuration, "sample_detector_distance", None),
            "notes": getattr(configuration, "notes", None),
        }
    )

config_rows


## 6. Inspect the dedicated reference stores

During initialization, `add_run()` also mirrors some reference runs into dedicated attributes:
- `dark`
- `empty_beam`
- `empty_cell`

This makes it easy to check what has already been recognized automatically.


In [ ]:
print("dark:")
for config_id, path in sorted(w.dark.items()):
    print(f"  {config_id}: {path}")

print("\nempty_beam:")
for config_id, modes in sorted(w.empty_beam.items()):
    print(f"  {config_id}: {modes}")

print("\nempty_cell:")
for config_id, modes in sorted(w.empty_cell.items()):
    print(f"  {config_id}: {modes}")


## 7. Review initialization issues

Warnings collected during initialization are often useful when tuning the automatic classification.


In [ ]:
issue_rows = [
    {
        "level": issue.level,
        "where": issue.where,
        "key": issue.key,
        "message": issue.message,
    }
    for issue in w.issues
]

issue_rows[:20]


## 8. Basic sanity checks

These checks keep the notebook focused on a simple initialization smoke test.


In [ ]:
assert len(w.runs) > 0, "Initialization should register at least one run"
assert len(w.configurations) > 0, "Initialization should detect at least one configuration"
assert all(path.exists() for path in w.runs.values()), "All registered run paths should exist"
assert w.output_dir == OUTPUT_DIR.resolve()

print("Initialization smoke test passed.")


In [ ]:
w.beam_centers